# Geometry first: automatic domain and conformal mesh

Define exact PEC/air shapes in your own coordinates. The domain is derived from the **final PEC bounds**. The default exterior uses 5 cells to TFSF, 4 to the integration contour, 6 to PML, then 12 PML cells per side. Exterior spacing is `C0 / fmax / 24`, independent of interior refinement.

This example constructs a deterministic mesh without an optimization run or reference solution. It repairs missed material intervals and missing enlarged-cell donors, then validates the actual conformal operator. An optional feasible_local optimization is provided below; leave RUN_OPTIMIZATION=False unless a qualified reference is available. Failure to construct within the limits is reported explicitly; geometric validity alone does not establish scattering accuracy.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import fdtdmesh as fd
from fdtdmesh.geometry_mesher import GeometryMeshingError

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT = ROOT / 'artifacts/notebooks/geometry_aware'
OUTPUT.mkdir(parents=True, exist_ok=True)
MESH_FIGSIZE = (14, 12)
RUN_GPU = True

sim = fd.Simulation(fmin=0.9e9, fmax=1.1e9)
# sim.add_circle((0.0, 0.0), 0.17, material='PEC')
# sim.add_circle((0.0756, -0.009), 0.15, material='air')
sim.add_ellipse((0.0, 0.0), (0.2,0.3), 0.0)
sim.add_ellipse((0.0, 0.0), (0.3,0.05), 0.0, material='air')
original_geometry = sim.geometry
print('Input PEC bounds [m]:', sim.geometry.bounds)
print('Derived domain [m]:', sim.size)
print('Solver coordinate offset [m]:', sim.coordinate_offset)
fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
sim.plot_geometry(units='wavelength', ax=ax)
plt.show()

## Construct and inspect

`target_spacing` is an interior resolution target, not a fixed cell budget. `max_cells` caps the total cells on each axis, including PML. Fixed exterior spacing can be larger than the interior target; transitions must meet the grading limit. The mesher may need extra cells to represent gaps and provide donors.

The original recipe remains available as `sim.geometry`; plots and solver arrays use `sim.computational_geometry`. Their coordinates differ by `sim.coordinate_offset`. The default phase origin is the centre of the final PEC bounding box in input coordinates, translated together with the geometry.

In [ ]:
mesh = None
try:
    mesh = sim.apply_mesh('geometry_aware', target_spacing=fd.C0 / sim.fmax / 32,
                          max_cells=(512, 512), max_passes=20, time_limit=30)
except GeometryMeshingError as exc:
    print(exc)
    display(exc.report)

if mesh is not None:
    assert sim.geometry == original_geometry
    print('Actual total cell counts:', (mesh.Nx, mesh.Ny))
    display(mesh.metadata['geometry_aware'])
    fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
    sim.plot_geometry(mesh=True, units='wavelength', ax=ax)
    ax.set_title(f'Geometry-aware mesh: {mesh.Nx} × {mesh.Ny}')
    fig.savefig(OUTPUT / 'mesh.png', dpi=150, bbox_inches='tight')
    plt.show()
    fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
    sim.plot_discretization(units='wavelength', ax=ax)
    ax.set_title('Conformal cut faces and enlarged-cell pairs')
    plt.show()
    sim.save(OUTPUT / 'simulation.h5')

## Optional GPU solve

The solver still performs field updates, DFT accumulation, convergence stopping, and NF2FF in native CUDA. This run demonstrates execution; use spatial and boundary convergence checks before treating its fields as a numerical reference.

In [ ]:
result = None
if RUN_GPU and mesh is not None:
    result = sim.solve()
    result.save(OUTPUT / 'result.h5')
    print('GPU status:', result.diagnostics['status'])
    assert result.input_geometry == original_geometry
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})
    result.plot_scattering(ax=axes[0])
    result.plot_far_field(component='magnitude', ax=axes[1])
    plt.show()

## Independent interior refinements, identical exterior

These are construction comparisons, not an accuracy study. The geometry, phase origin, physical boxes, PML thickness, and reserved exterior axis lines stay fixed. Lines associated with the interior still extend tangentially across the domain because the solver uses a tensor-product grid.

In [ ]:
for ppw in (40, 48):
    try:
        comparison = fd.Simulation(fmin=sim.fmin, fmax=sim.fmax)
        comparison.set_geometry(original_geometry)
        other = comparison.apply_mesh('geometry_aware', target_spacing=fd.C0 / sim.fmax / ppw,
                                      max_cells=(512, 512), time_limit=30)
        assert comparison.layout == sim.layout
        if mesh is not None:
            for a, b in ((mesh.x, other.x), (mesh.y, other.y)):
                np.testing.assert_allclose(a[:28], b[:28], rtol=0, atol=1e-14)
                np.testing.assert_allclose(a[-28:], b[-28:], rtol=0, atol=1e-14)
        fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
        comparison.plot_geometry(mesh=True, units='wavelength', ax=ax)
        ax.set_title(f'Interior target λmin/{ppw}: {other.Nx} × {other.Ny}')
        plt.show()
    except GeometryMeshingError as exc:
        print(f'λmin/{ppw}:', exc)
        display(exc.report)

## Optional feasible-local optimization

Set `RUN_OPTIMIZATION=True` to qualify a numerical reference and run the opt-in `feasible_local` search from the validated geometry-aware seed. Every candidate keeps the seed's exact `(Nx, Ny)` cell counts and exact geometry. `analyze_mesh_adaptivity` is an optional CPU-only diagnostic for conditional LP coordinate ranges; its ranges omit nonlinear geometry and donor constraints. The optimizer's stricter winner validation runs independently after the search.

In [ ]:
RUN_OPTIMIZATION = False
optimization = None
reference = None

if RUN_OPTIMIZATION and mesh is not None:
    from fdtdmesh.benchmarks import optimize_mesh, qualify_reference

    reference = qualify_reference(sim, directory=OUTPUT / 'reference')
    if reference.qualified:
        optimization = optimize_mesh(
            sim,
            reference,
            initial_mesh=mesh,
            cells=(mesh.Nx, mesh.Ny),
            strategy='feasible_local',
            directory=OUTPUT / 'optimization',
            max_evaluations=200,
            max_solves=120,
            max_seconds=600,
        )
        display(optimization.report.get('search_statistics', {}))
        if optimization.best is not None:
            sim.apply_mesh(optimization.best.mesh)
            fig, ax = plt.subplots(figsize=(18, 16), dpi=120)
            sim.plot_geometry(mesh=True, units='wavelength', ax=ax)
            ax.set_title(f'Best feasible-local geometry mesh: {optimization.best.mesh.Nx} × {optimization.best.mesh.Ny}')
            plt.show()
            display(optimization.plot_history())
    else:
        print('Reference was not qualified; optimization skipped.')
elif RUN_OPTIMIZATION:
    print('Geometry-aware seed unavailable; optimization skipped.')